# Silver Layer: ERP Product Category CDC

**Source**: `workspace.bronze.erp_pxcat_cdc`  
**Target**: `workspace.silver.erp_product_category_cdc`  
**Primary Key**: `id` → `category_id`

**Transformations:**
1. Trim strings
2. Normalize maintenance flag to boolean
3. Rename columns

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, trim, when, upper, lit
from delta.tables import DeltaTable

bronze_table = "workspace.bronze.erp_px_cat_g1v2_cdc"
silver_table = "workspace.silver.erp_product_category_cdc"

print("✓ Configuration loaded")

In [0]:
silver_exists = spark.catalog.tableExists(silver_table)

if silver_exists:
    watermark = spark.sql(f"""
        SELECT COALESCE(MAX(ingestion_timestamp), '1900-01-01') as max_timestamp
        FROM {silver_table}
    """).collect()[0]["max_timestamp"]
    print(f"✓ Watermark: {watermark}")
else:
    watermark = "1900-01-01"
    print(f"ℹ️  Initial load")

In [0]:
df = spark.table(bronze_table).filter(col("ingestion_timestamp") > watermark)

new_count = df.count()
print(f"📊 New records: {new_count:,}")

if new_count == 0:
    dbutils.notebook.exit("No new data")

In [0]:
# Trim strings
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

print("✓ Trimmed")

In [0]:
# Normalize maintenance flag to boolean
df = df.withColumn(
    "maintenance",
    when(upper(col("maintenance")) == "YES", lit(True))
    .when(upper(col("maintenance")) == "NO", lit(False))
    .otherwise(None)
)

print("✓ Normalized maintenance flag")

In [0]:
# Rename columns
rename_map = {
    "id": "category_id",
    "cat": "category",
    "subcat": "subcategory",
    "maintenance": "maintenance_flag"
}

for old_name, new_name in rename_map.items():
    df = df.withColumnRenamed(old_name, new_name)

print("✓ Renamed columns")
df.limit(5).display()

In [0]:
if not silver_exists:
    print("Creating Silver table...")
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(silver_table))
    print(f"✓ Created: {spark.table(silver_table).count():,} records")
else:
    print("Merging into Silver...")
    target = DeltaTable.forName(spark, silver_table)
    (target.alias("target")
      .merge(df.alias("source"), "target.category_id = source.category_id")
      .whenMatchedUpdateAll()
      .whenNotMatchedInsertAll()
      .execute())
    print(f"✓ MERGE complete: {spark.table(silver_table).count():,} total")